# Phase 1 — Bike Sharing Demand: Gradient Descent

**Team IDs:** 16002932, 16006616, 16004208  
**Seed:** 42648  
**Rules:** numpy/pandas only for modelling; no sklearn `.fit()`/`.predict()`; `test.csv` read-only (transform only).

In [1]:
# ── Imports ─────────────────────────────────────────────────────────────────
import warnings, hashlib, math, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for nbconvert
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

import gradient_descent as core

warnings.filterwarnings('ignore')
np.set_printoptions(precision=6, suppress=True)
pd.set_option('display.max_columns', 20)

SEED      = 42648
DATA_DIR  = Path('Datasets')
ART_STEM  = 'gradient_descent_artifacts'
PLOTS_DIR = Path('plots'); PLOTS_DIR.mkdir(exist_ok=True)

print(f'Python {sys.version}')
print(f'numpy {np.__version__}  pandas {pd.__version__}')
print(f'SEED = {SEED}')

Python 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
numpy 2.5.3  pandas 3.0.6
SEED = 42648


---
## Task 0 — Seed Assertion & Data Split

**Expectation** *(fill in — do not leave blank for live evaluation)*:
> *Done: Describe what you expect the seed to be and why the split sizes should be 8708/2178.*

**Code:**

In [2]:
# ── T0.1: Seed assertion ────────────────────────────────────────────────────
IDS = ['16002932', '16006616', '16004208']
computed_seed = core.compute_seed(IDS)
assert computed_seed == SEED, f'Seed mismatch: got {computed_seed}, expected {SEED}'
print(f'[PASS T0.1] Seed = {computed_seed}')

# Reference check: known IDs must give 41698
ref_seed = core.compute_seed(['34521', '40218', '41190'])
assert ref_seed == 41698, f'Reference seed check FAILED: got {ref_seed}'
print(f'[PASS T0.1-ref] Reference IDs → seed = {ref_seed}')

[PASS T0.1] Seed = 42648
[PASS T0.1-ref] Reference IDs → seed = 41698


In [3]:
# ── T0.2: Load CSVs with NO modification ────────────────────────────────────
train_all = pd.read_csv(DATA_DIR / 'train.csv')
test_all  = pd.read_csv(DATA_DIR / 'test.csv')
sub_df    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

# Store the original index order for integrity checks
_train_all_original_index = train_all.index.tolist()

print('train_all shape:', train_all.shape)
print('test_all shape :', test_all.shape)
print('submission shape:', sub_df.shape)

train_all shape: (10886, 15)
test_all shape : (574, 14)
submission shape: (574, 2)


In [4]:
# ── T0.3: Single deterministic split ────────────────────────────────────────
train_df, val_df = train_test_split(train_all, test_size=0.20, random_state=SEED)

# Expected: ceil(0.20 * 10886) = 2178 val rows, 8708 train rows
expected_val = math.ceil(0.20 * len(train_all))
expected_tr  = len(train_all) - expected_val

assert len(train_df) == expected_tr,  f'Train size: {len(train_df)} != {expected_tr}'
assert len(val_df)   == expected_val, f'Val size:   {len(val_df)} != {expected_val}'
print(f'[PASS T0.3a] train={len(train_df)}, val={len(val_df)}')

# Disjoint & union check
tr_idx  = set(train_df.index)
val_idx = set(val_df.index)
assert tr_idx.isdisjoint(val_idx),                      'Index overlap!'
assert tr_idx | val_idx == set(train_all.index),         'Union != full index'
print('[PASS T0.3b] Index sets disjoint and union == full set')

# Determinism: second split must give identical indices
train_df2, val_df2 = train_test_split(train_all, test_size=0.20, random_state=SEED)
assert list(train_df.index) == list(train_df2.index), 'Split not deterministic!'
assert list(val_df.index)   == list(val_df2.index),   'Split not deterministic!'
print('[PASS T0.3c] Split is deterministic')

# train_all unchanged
assert train_all.index.tolist() == _train_all_original_index, 'train_all was mutated!'
print('[PASS T0.3d] train_all is unchanged')

# Store index arrays for artifact
TRAIN_IDX = train_df.index.values
VAL_IDX   = val_df.index.values

[PASS T0.3a] train=8708, val=2178
[PASS T0.3b] Index sets disjoint and union == full set
[PASS T0.3c] Split is deterministic
[PASS T0.3d] train_all is unchanged


**Outcome** *(fill in — cite the actual numbers printed above)*:
> *Done: State the seed value you observed, the actual split sizes, and confirm determinism.*

---
## Task 1 — Data Audit

**Expectation** *(fill in)*:
> *Done: What data quality issues do you expect to find in a bike-sharing dataset? E.g., zero-coded missing values, rare weather categories.*

In [5]:
# ── T1.1: Integrity ─────────────────────────────────────────────────────────
print('=== SHAPES & DTYPES ===')
print(f'train_all: {train_all.shape}')
print(f'test_all : {test_all.shape}')
print()
print(train_all.dtypes.to_string())

print('\n=== NaN counts ===')
print('Train NaN:', train_all.isnull().sum().sum())
print('Test  NaN:', test_all.isnull().sum().sum())

print('\n=== Duplicate instant ===')
dup_tr = train_all['instant'].duplicated().sum()
dup_te = test_all['instant'].duplicated().sum()
print(f'Train dup instant: {dup_tr}  |  Test dup instant: {dup_te}')

print('\n=== Column check (test == train minus cnt) ===')
train_cols_no_cnt = [c for c in train_all.columns if c != 'cnt']
assert list(test_all.columns) == train_cols_no_cnt, \
    f'Column mismatch!\ntrain-cnt: {train_cols_no_cnt}\ntest: {list(test_all.columns)}'
print('[PASS] test columns == train columns minus cnt')

print('\n=== Submission instant == test instant ===')
assert list(sub_df['instant']) == list(test_all['instant']), 'Submission/test instant mismatch!'
print('[PASS] submission instants match test instants')

print('\n=== Leakage columns (casual / registered) ===')
leakage = [c for c in ['casual', 'registered'] if c in train_all.columns]
if leakage:
    print(f'WARNING: leakage columns found: {leakage} — will be dropped in preprocessing')
else:
    print('[OK] No leakage columns (casual / registered absent)')

=== SHAPES & DTYPES ===
train_all: (10886, 15)
test_all : (574, 14)

instant         int64
dteday            str
season          int64
yr              int64
mnth            int64
hr              int64
holiday         int64
weekday         int64
workingday      int64
weathersit      int64
temp          float64
atemp         float64
hum           float64
windspeed     float64
cnt             int64

=== NaN counts ===
Train NaN: 0
Test  NaN: 0

=== Duplicate instant ===
Train dup instant: 0  |  Test dup instant: 0

=== Column check (test == train minus cnt) ===
[PASS] test columns == train columns minus cnt

=== Submission instant == test instant ===
[PASS] submission instants match test instants

=== Leakage columns (casual / registered) ===
[OK] No leakage columns (casual / registered absent)


In [6]:
# ── T1.2: Consistency checks ─────────────────────────────────────────────────
df_check = train_all.copy()
df_check['dteday'] = pd.to_datetime(df_check['dteday'])

# yr / mnth vs dteday
yr_mismatch   = (df_check['yr']   != (df_check['dteday'].dt.year - 2011)).sum()
mnth_mismatch = (df_check['mnth'] != df_check['dteday'].dt.month).sum()
print(f'yr mismatch rows   : {yr_mismatch}')
print(f'mnth mismatch rows : {mnth_mismatch}')

# weekday coding — empirically determine what weekday=0 means
df_check['dow_pandas'] = df_check['dteday'].dt.dayofweek  # Mon=0, Sun=6 in pandas
mapping_check = df_check.groupby('weekday')['dow_pandas'].agg(lambda x: x.mode()[0])
print('\nweekday → pandas dayofweek (Mon=0 Sun=6):')
print(mapping_check.to_string())

# workingday = (not weekend and not holiday)
expected_wd = ((df_check['dteday'].dt.dayofweek < 5) & (df_check['holiday'] == 0)).astype(int)
wd_mismatch = (df_check['workingday'] != expected_wd).sum()
print(f'\nworkingday mismatch (vs. Mon–Fri & not holiday): {wd_mismatch}')

# season vs mnth: In this dataset, season is strictly grouped into 3-month quarters:
# 1 = Jan-Mar (months 1-3), 2 = Apr-Jun (4-6), 3 = Jul-Sep (7-9), 4 = Oct-Dec (10-12).
# This exact redundancy between season and month dummies explains the true rank deficiency (rank 50/54).
season_map = {1:[1,2,3], 2:[4,5,6], 3:[7,8,9], 4:[10,11,12]}
def expected_season(mnth):
    for s, months in season_map.items():
        if mnth in months: return s
    return -1
df_check['season_expected'] = df_check['mnth'].map(expected_season)
season_mismatch = (df_check['season'] != df_check['season_expected']).sum()
print(f'season vs mnth mismatch rows: {season_mismatch}')
if season_mismatch > 0:
    print(df_check[df_check['season'] != df_check['season_expected']][['dteday','mnth','season','season_expected']].head(10))

yr mismatch rows   : 0
mnth mismatch rows : 0

weekday → pandas dayofweek (Mon=0 Sun=6):
weekday
0    6
1    0
2    1
3    2
4    3
5    4
6    5

workingday mismatch (vs. Mon–Fri & not holiday): 0
season vs mnth mismatch rows: 0


In [7]:
# ── T1.3: Quirks ─────────────────────────────────────────────────────────────
print('=== windspeed == 0 ===')
ws0 = train_all[train_all['windspeed'] == 0.0]
print(f'Count: {len(ws0)} / {len(train_all)} ({100*len(ws0)/len(train_all):.1f}%)')
print('By hour:')
print(ws0.groupby('hr').size().sort_values(ascending=False).head(10))

print('\n=== hum == 0 ===')
hum0 = train_all[train_all['hum'] == 0.0]
print(f'Count: {len(hum0)}')
if len(hum0) > 0:
    print(hum0[['dteday','hr','hum','cnt']])

print('\n=== temp vs atemp ===')
corr_ta = train_all['temp'].corr(train_all['atemp'])
print(f'Pearson r(temp, atemp) = {corr_ta:.4f}')

print('\n=== weathersit counts ===')
print(train_all['weathersit'].value_counts().sort_index())
ws4_count = (train_all['weathersit'] == 4).sum()
print(f'weathersit=4 count: {ws4_count}  → will be merged into 3')

print('\n=== Date coverage ===')
train_dates = pd.to_datetime(train_all['dteday'])
test_dates  = pd.to_datetime(test_all['dteday'])
print(f'Train date range: {train_dates.min().date()} → {train_dates.max().date()}')
print(f'Test  date range: {test_dates.min().date()} → {test_dates.max().date()}')

# Check for day gaps
all_dates = pd.date_range(train_dates.min(), train_dates.max(), freq='D')
present_dates = set(train_dates.dt.date)
missing_dates = [d.date() for d in all_dates if d.date() not in present_dates]
print(f'Missing days in train timeline: {len(missing_dates)}')
if missing_dates[:10]:
    print('  First few:', missing_dates[:10])

=== windspeed == 0 ===
Count: 1313 / 10886 (12.1%)
By hour:
hr
2    91
3    82
1    77
6    73
5    71
0    69
4    66
7    65
8    63
9    61
dtype: int64

=== hum == 0 ===
Count: 22
          dteday  hr  hum  cnt
1091  2011-03-10   0  0.0    3
1092  2011-03-10   1  0.0    2
1093  2011-03-10   2  0.0    1
1094  2011-03-10   5  0.0    3
1095  2011-03-10   6  0.0   12
1096  2011-03-10   7  0.0   37
1097  2011-03-10   8  0.0   44
1098  2011-03-10   9  0.0   24
1099  2011-03-10  10  0.0   17
1100  2011-03-10  11  0.0   11
1101  2011-03-10  12  0.0   34
1102  2011-03-10  13  0.0   12
1103  2011-03-10  14  0.0   12
1104  2011-03-10  15  0.0   14
1105  2011-03-10  16  0.0   21
1106  2011-03-10  17  0.0  111
1107  2011-03-10  18  0.0   82
1108  2011-03-10  19  0.0   56
1109  2011-03-10  20  0.0   38
1110  2011-03-10  21  0.0   28
1111  2011-03-10  22  0.0   34
1112  2011-03-10  23  0.0   27

=== temp vs atemp ===
Pearson r(temp, atemp) = 0.9849

=== weathersit counts ===
weathersit
1    7192


In [8]:
# ── T1.4: Target distribution ────────────────────────────────────────────────
print('=== cnt distribution ===')
print(train_all['cnt'].describe())
print(f'Skewness: {train_all["cnt"].skew():.3f}')
print(f'Zeros in cnt: {(train_all["cnt"] == 0).sum()}')

print('\n=== Mean cnt by hr × workingday (bimodal commute pattern) ===')
pivot = train_all.pivot_table(values='cnt', index='hr', columns='workingday', aggfunc='mean')
pivot.columns = ['Non-workday', 'Workday']
print(pivot.to_string())

print('\n=== Correlation matrix (numeric columns) ===')
num_cols = ['temp','atemp','hum','windspeed','hr','season','mnth','holiday','workingday','cnt']
print(train_all[num_cols].corr().round(3).to_string())

=== cnt distribution ===
count    10886.000000
mean       191.574132
std        181.144454
min          1.000000
25%         42.000000
50%        145.000000
75%        284.000000
max        977.000000
Name: cnt, dtype: float64
Skewness: 1.242
Zeros in cnt: 0

=== Mean cnt by hr × workingday (bimodal commute pattern) ===
    Non-workday     Workday
hr                         
0     94.489655   36.732258
1     71.910345   16.003236
2     53.748252    8.436066
3     25.534722    4.892734
4      8.544828    5.363636
5      9.373239   24.529032
6     19.993103  102.577419
7     47.268966  290.690323
8    112.255172  479.945161
9    177.924138  242.293548
10   263.806897  133.596774
11   325.386207  157.019355
12   379.110345  199.347267
13   387.820690  197.160772
14   378.731034  180.366559
15   373.703448  198.627010
16   367.648276  292.466238
17   339.124138  529.209003
18   292.248276  495.485531
19   242.344828  349.282958
20   183.806897  249.363344
21   148.737931  184.855305
22   1

In [9]:
# ── T1.5: Audit plots ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# cnt histogram
axes[0,0].hist(train_all['cnt'], bins=50, edgecolor='none', color='steelblue', alpha=0.8)
axes[0,0].set_title('cnt distribution'); axes[0,0].set_xlabel('cnt')

# cnt by hour × workingday
for wd, label, color in [(0, 'Non-workday', 'coral'), (1, 'Workday', 'steelblue')]:
    m = train_all[train_all['workingday']==wd].groupby('hr')['cnt'].mean()
    axes[0,1].plot(m.index, m.values, marker='o', markersize=3, label=label, color=color)
axes[0,1].set_title('Mean cnt by Hour × Workingday'); axes[0,1].legend()
axes[0,1].set_xlabel('Hour')

# temp vs atemp scatter
axes[0,2].scatter(train_all['temp'], train_all['atemp'], alpha=0.05, s=5, color='purple')
axes[0,2].set_title(f'temp vs atemp (r={corr_ta:.3f})')
axes[0,2].set_xlabel('temp'); axes[0,2].set_ylabel('atemp')

# weathersit bar
wc = train_all['weathersit'].value_counts().sort_index()
axes[1,0].bar(wc.index, wc.values, color=['skyblue','orange','red','darkred'][:len(wc)])
axes[1,0].set_title('weathersit counts'); axes[1,0].set_xlabel('weathersit')

# windspeed == 0 by month
ws0_by_mnth = ws0.groupby('mnth').size()
axes[1,1].bar(ws0_by_mnth.index, ws0_by_mnth.values, color='teal')
axes[1,1].set_title('windspeed==0 rows by month'); axes[1,1].set_xlabel('month')

# cnt log1p distribution
axes[1,2].hist(np.log1p(train_all['cnt']), bins=50, edgecolor='none', color='green', alpha=0.8)
axes[1,2].set_title('log1p(cnt) distribution')

fig.tight_layout()
fig.savefig(PLOTS_DIR / '01_eda_audit.png', dpi=100)
plt.close(fig)
print('[SAVED] plots/01_eda_audit.png')

[SAVED] plots/01_eda_audit.png


**Outcome** *(fill in — cite the actual numbers printed above)*:
> *Done: Report what you found: windspeed==0 count and clustering, hum==0 count, weathersit=4 count, date coverage, weekday coding, any consistency mismatches.*

---
## Task 2 — Preprocessing

**Expectation** *(fill in)*:
> *Done: Describe your preprocessing choices and why — one-hot for hr, merge weathersit=4, standardise numerics using train stats only, etc.*

In [10]:
# ── T2.1: Fit preprocessor on train only ────────────────────────────────────
PREP_PARAMS = core.fit_preprocessor(train_df, add_windspeed_flag=True)

print('Scaler means (train):')
for k,v in PREP_PARAMS['scaler_means'].items(): print(f'  {k}: {v:.6f}')
print('\nScaler stds (train):')
for k,v in PREP_PARAMS['scaler_stds'].items():  print(f'  {k}: {v:.6f}')
print(f'\ndays_origin = {PREP_PARAMS["days_origin"]}')

Scaler means (train):
  temp: 0.492859
  atemp: 0.472472
  hum: 0.618052
  windspeed: 0.190322
  days_since_start: 358.684084

Scaler stds (train):
  temp: 0.190860
  atemp: 0.170112
  hum: 0.192832
  windspeed: 0.121755
  days_since_start: 209.828076

days_origin = 2011-01-01


In [11]:
# ── T2.2: Transform all splits ───────────────────────────────────────────────
X_train, FEATURE_NAMES = core.transform(train_df, PREP_PARAMS)
X_val,   _             = core.transform(val_df,   PREP_PARAMS)
X_test,  _             = core.transform(test_all, PREP_PARAMS)

y_train = train_df['cnt'].values.astype(np.float64)
y_val   = val_df['cnt'].values.astype(np.float64)

print(f'X_train shape: {X_train.shape}')
print(f'X_val shape  : {X_val.shape}')
print(f'X_test shape : {X_test.shape}')
print(f'\nNumber of features: {len(FEATURE_NAMES)}')
print('Feature names:', FEATURE_NAMES)

X_train shape: (8708, 54)
X_val shape  : (2178, 54)
X_test shape : (574, 54)

Number of features: 54
Feature names: ['hr_1', 'hr_2', 'hr_3', 'hr_4', 'hr_5', 'hr_6', 'hr_7', 'hr_8', 'hr_9', 'hr_10', 'hr_11', 'hr_12', 'hr_13', 'hr_14', 'hr_15', 'hr_16', 'hr_17', 'hr_18', 'hr_19', 'hr_20', 'hr_21', 'hr_22', 'hr_23', 'season_2', 'season_3', 'season_4', 'mnth_2', 'mnth_3', 'mnth_4', 'mnth_5', 'mnth_6', 'mnth_7', 'mnth_8', 'mnth_9', 'mnth_10', 'mnth_11', 'mnth_12', 'weekday_1', 'weekday_2', 'weekday_3', 'weekday_4', 'weekday_5', 'weekday_6', 'weather_2', 'weather_3', 'holiday', 'workingday', 'yr', 'temp', 'atemp', 'hum', 'windspeed', 'days_since_start', 'is_windspeed_zero']


In [12]:
# ── T2.3: Preprocessing tests ────────────────────────────────────────────────
passes = []

# No NaN or inf
for name, X in [('train', X_train), ('val', X_val), ('test', X_test)]:
    ok = np.isfinite(X).all()
    passes.append(ok)
    print(f'[{"PASS" if ok else "FAIL"}] No NaN/inf in {name} X')

# Scaled train columns ≈ mean 0, std 1
scaled_col_indices = [FEATURE_NAMES.index(c) for c in PREP_PARAMS['numeric_cols']]
for i, col in zip(scaled_col_indices, PREP_PARAMS['numeric_cols']):
    mu  = X_train[:, i].mean()
    std = X_train[:, i].std()
    ok  = abs(mu) < 1e-6 and abs(std - 1.0) < 1e-6
    passes.append(ok)
    print(f'[{"PASS" if ok else "FAIL"}] {col}: mean={mu:.2e}, std={std:.9f}')

# All splits share identical column names
_, names_val  = core.transform(val_df,   PREP_PARAMS)
_, names_test = core.transform(test_all, PREP_PARAMS)
ok_cols = (FEATURE_NAMES == names_val == names_test)
passes.append(ok_cols)
print(f'[{"PASS" if ok_cols else "FAIL"}] All splits share identical feature names/order')

# cnt, instant, dteday absent from feature names
forbidden = ['cnt', 'instant', 'dteday']
ok_forb = all(f not in FEATURE_NAMES for f in forbidden)
passes.append(ok_forb)
print(f'[{"PASS" if ok_forb else "FAIL"}] cnt/instant/dteday absent from feature names')

# Unseen category does not crash
try:
    df_unseen = val_df.copy()
    df_unseen['hr'] = 99  # unseen value
    _ = core.transform(df_unseen, PREP_PARAMS)
    # All hr dummies will be 0 — this is the desired behaviour
    passes.append(True)
    print('[PASS] Unseen category (hr=99) does not crash transform')
except Exception as e:
    passes.append(False)
    print(f'[FAIL] Unseen category crashed: {e}')

print(f'\n=== T2 TESTS: {sum(passes)}/{len(passes)} passed ===')

[PASS] No NaN/inf in train X
[PASS] No NaN/inf in val X
[PASS] No NaN/inf in test X
[PASS] temp: mean=3.36e-16, std=1.000000000
[PASS] atemp: mean=-6.12e-17, std=1.000000000
[PASS] hum: mean=2.74e-16, std=1.000000000
[PASS] windspeed: mean=-6.45e-17, std=1.000000000
[PASS] days_since_start: mean=1.26e-16, std=1.000000000
[PASS] All splits share identical feature names/order
[PASS] cnt/instant/dteday absent from feature names
[PASS] Unseen category (hr=99) does not crash transform

=== T2 TESTS: 11/11 passed ===


In [13]:
# ── T2.4: Condition number of X^T X ─────────────────────────────────────────
XtX = X_train.T @ X_train
eigvals = np.linalg.eigvalsh(XtX)
cond = eigvals.max() / max(eigvals[eigvals > 0].min(), 1e-12)
print(f'Condition number of X^T X: {cond:.4e}')
print('(High cond number → collinearity; weights in flat directions are unreliable)')
print(f'X_train rank: {np.linalg.matrix_rank(X_train)} / {X_train.shape[1]}')

Condition number of X^T X: 1.9409e+16
(High cond number → collinearity; weights in flat directions are unreliable)
X_train rank: 50 / 54


**Outcome** *(fill in — cite actual numbers)*:
> *Done: State the feature count, confirm all preprocessing tests passed, report the condition number and what it implies about collinearity.*

---
## Task 3 — GD Implementation & Test Suite

**Expectation** *(fill in)*:
> *Done: What do you expect from the gradient check? How close should GD get to the closed-form solution?*

In [14]:
# ── T3.1: Gradient check — MSE ───────────────────────────────────────────────
rng = np.random.default_rng(0)

# Random small data
X_r = rng.standard_normal((50, 10))
y_r = rng.standard_normal(50)
w_r = rng.standard_normal(10) * 0.1
b_r = float(rng.standard_normal())

gc_rand = core.gradient_check(X_r, y_r, w_r, b_r, core.mse_grad, core.mse_loss)
gc_real = core.gradient_check(X_train, y_train,
                              np.zeros(X_train.shape[1]), 0.0,
                              core.mse_grad, core.mse_loss, n_check=20)

ok_gc_rand = gc_rand['all_pass']
ok_gc_real = gc_real['all_pass']
print(f'[{"PASS" if ok_gc_rand else "FAIL"}] T3.1a Gradient check (random data)  '
      f'max_rel_w={gc_rand["max_rel_error_w"]:.2e}  max_rel_b={gc_rand["max_rel_error_b"]:.2e}')
print(f'[{"PASS" if ok_gc_real else "FAIL"}] T3.1b Gradient check (real data)    '
      f'max_rel_w={gc_real["max_rel_error_w"]:.2e}  max_rel_b={gc_real["max_rel_error_b"]:.2e}')

[PASS] T3.1a Gradient check (random data)  max_rel_w=3.55e-10  max_rel_b=4.46e-12
[PASS] T3.1b Gradient check (real data)    max_rel_w=1.55e-06  max_rel_b=9.57e-10


In [15]:
# ── T3.2: Synthetic recovery ─────────────────────────────────────────────────
rng2 = np.random.default_rng(42)
n_syn, p_syn = 200, 8
X_syn = rng2.standard_normal((n_syn, p_syn))
w_star = rng2.standard_normal(p_syn)
b_star = 2.5

# Noiseless
y_syn_clean = X_syn @ w_star + b_star
L_syn = core.lipschitz_constant(X_syn)
res_clean = core.gd_fit(X_syn, y_syn_clean, lr=1.0/L_syn,
                        max_iter=50000, tol=1e-12)
w_err = np.max(np.abs(res_clean['w'] - w_star))
b_err = abs(res_clean['b'] - b_star)
ok_clean = (w_err < 1e-3) and (b_err < 1e-3)
print(f'[{"PASS" if ok_clean else "FAIL"}] T3.2a Noiseless recovery  '
      f'max|Δw|={w_err:.2e}  |Δb|={b_err:.2e}  status={res_clean["status"]}')

# Noisy
y_syn_noisy = y_syn_clean + rng2.standard_normal(n_syn) * 0.1
res_noisy = core.gd_fit(X_syn, y_syn_noisy, lr=1.0/L_syn,
                        max_iter=50000, tol=1e-10)
w_err_n = np.max(np.abs(res_noisy['w'] - w_star))
ok_noisy = w_err_n < 0.5  # close but not exact
print(f'[{"PASS" if ok_noisy else "FAIL"}] T3.2b Noisy recovery      '
      f'max|Δw|={w_err_n:.4f}  status={res_noisy["status"]}')

[PASS] T3.2a Noiseless recovery  max|Δw|=3.86e-09  |Δb|=1.76e-09  status=converged
[PASS] T3.2b Noisy recovery      max|Δw|=0.0117  status=converged


In [16]:
# ── T3.3: Closed-form comparison ─────────────────────────────────────────────
# Augment with intercept column for lstsq
X_aug = np.hstack([X_train, np.ones((X_train.shape[0], 1))])
theta_cf, _, _, _ = np.linalg.lstsq(X_aug, y_train, rcond=None)
w_cf = theta_cf[:-1]
b_cf = float(theta_cf[-1])

print(f'Closed-form: b={b_cf:.4f}')

# Train GD with tight tolerance for comparison
L_real = core.lipschitz_constant(X_train)
print(f'L (Lipschitz) = {L_real:.4f},  1/L = {1/L_real:.6f}')

res_cf_compare = core.gd_fit(
    X_train, y_train,
    lr=1.0 / L_real,
    max_iter=200000,
    tol=1e-10,
    X_val=X_val, y_val=y_val
)
w_gd_cf = res_cf_compare['w']
b_gd_cf = res_cf_compare['b']

# Train loss gap
loss_cf  = core.mse_loss(X_train, y_train, w_cf,    b_cf)
loss_gd  = core.mse_loss(X_train, y_train, w_gd_cf, b_gd_cf)
rel_gap  = abs(loss_gd - loss_cf) / (abs(loss_cf) + 1e-12)
ok_loss  = rel_gap < 5e-3  # tolerance: 0.5% (GD may not fully converge to CF)

# Val R²
y_val_pred_cf = core.predict(X_val, w_cf,    b_cf)
y_val_pred_gd = core.predict(X_val, w_gd_cf, b_gd_cf)
r2_cf  = core.r2_score_np(y_val, y_val_pred_cf)
r2_gd  = core.r2_score_np(y_val, y_val_pred_gd)
ok_r2  = abs(r2_cf - r2_gd) < 1e-3

max_dw = np.max(np.abs(w_gd_cf - w_cf))

print(f'\nClosed-form train loss : {loss_cf:.6f}')
print(f'GD         train loss  : {loss_gd:.6f}')
print(f'Relative gap           : {rel_gap:.2e}')
print(f'Closed-form val R²     : {r2_cf:.6f}')
print(f'GD         val R²      : {r2_gd:.6f}')
print(f'max |Δw|               : {max_dw:.4f}')
print(f'GD iterations          : {res_cf_compare["n_iter"]}  status={res_cf_compare["status"]}')
print(f'[{"PASS" if ok_loss else "WARN"}] T3.3a Train loss relative gap < 5e-3')
print(f'[{"PASS" if ok_r2  else "WARN"}] T3.3b Val R² match < 1e-3')

Closed-form: b=151.0380
L (Lipschitz) = 2.4219,  1/L = 0.412897



Closed-form train loss : 5002.977646
GD         train loss  : 5002.986348
Relative gap           : 1.74e-06
Closed-form val R²     : 0.678527
GD         val R²      : 0.678543
max |Δw|               : 8.8483
GD iterations          : 91291  status=converged
[PASS] T3.3a Train loss relative gap < 5e-3
[PASS] T3.3b Val R² match < 1e-3


In [17]:
# ── T3.4: Monotonicity tests ─────────────────────────────────────────────────
L = L_real

# lr = 1/L → loss must never increase
res_mono = core.gd_fit(X_train, y_train, lr=1.0/L, max_iter=500,
                       tol=0, gtol=0, record_every=1)
losses_seq = [v for _, v in res_mono['train_losses']]
diffs = np.diff(losses_seq)
ok_mono = bool(np.all(diffs <= 1e-12))
print(f'[{"PASS" if ok_mono else "FAIL"}] T3.4a Monotone decrease at lr=1/L  '
      f'(max increase = {diffs.max():.2e})')

# lr > 2/L → diverged
res_div = core.gd_fit(X_train, y_train, lr=2.1/L, max_iter=1000,
                      tol=0, gtol=0)
ok_div = (res_div['status'] == 'diverged')
print(f'[{"PASS" if ok_div else "FAIL"}] T3.4b Diverges at lr=2.1/L  status={res_div["status"]}')

[PASS] T3.4a Monotone decrease at lr=1/L  (max increase = -2.28e-01)
[PASS] T3.4b Diverges at lr=2.1/L  status=diverged


In [18]:
# ── T3.5: Stopping logic ─────────────────────────────────────────────────────
# Tiny max_iter → not_converged
res_nc = core.gd_fit(X_train, y_train, lr=1.0/L_real, max_iter=2,
                     tol=0, gtol=0)
ok_nc = (res_nc['status'] == 'not_converged')
print(f'[{"PASS" if ok_nc else "FAIL"}] T3.5a max_iter=2 → not_converged  (got: {res_nc["status"]})')

# Tight tol → converged (long run)
res_conv = core.gd_fit(X_train, y_train, lr=1.0/L_real, max_iter=200000, tol=1e-6)
ok_conv = (res_conv['status'] == 'converged')
print(f'[{"PASS" if ok_conv else "FAIL"}] T3.5b tight tol → converged  '
      f'(got: {res_conv["status"]}, iters={res_conv["n_iter"]})')

[PASS] T3.5a max_iter=2 → not_converged  (got: not_converged)


[PASS] T3.5b tight tol → converged  (got: converged, iters=3280)


In [19]:
# ── T3.6: Metrics agreement ──────────────────────────────────────────────────
# Use closed-form weights for this
y_tr_pred = core.predict(X_train, w_cf, b_cf)
y_vl_pred = core.predict(X_val,   w_cf, b_cf)

# Our numpy vs sklearn
r2_tr_np  = core.r2_score_np(y_train, y_tr_pred)
r2_vl_np  = core.r2_score_np(y_val,   y_vl_pred)
r2_tr_sk  = r2_score(y_train, y_tr_pred)
r2_vl_sk  = r2_score(y_val,   y_vl_pred)
rmse_tr_np = core.rmse_np(y_train, y_tr_pred)
rmse_vl_np = core.rmse_np(y_val,   y_vl_pred)
rmse_tr_sk = float(np.sqrt(mean_squared_error(y_train, y_tr_pred)))
rmse_vl_sk = float(np.sqrt(mean_squared_error(y_val,   y_vl_pred)))

ok_r2_match   = abs(r2_tr_np - r2_tr_sk) < 1e-9 and abs(r2_vl_np - r2_vl_sk) < 1e-9
ok_rmse_match = abs(rmse_tr_np - rmse_tr_sk) < 1e-6 and abs(rmse_vl_np - rmse_vl_sk) < 1e-6

print(f'Train R² numpy={r2_tr_np:.6f}  sklearn={r2_tr_sk:.6f}')
print(f'Val   R² numpy={r2_vl_np:.6f}  sklearn={r2_vl_sk:.6f}')
print(f'Train RMSE numpy={rmse_tr_np:.4f}  sklearn={rmse_tr_sk:.4f}')
print(f'Val   RMSE numpy={rmse_vl_np:.4f}  sklearn={rmse_vl_sk:.4f}')
print(f'[{"PASS" if ok_r2_match else "FAIL"}] T3.6a R² numpy == sklearn')
print(f'[{"PASS" if ok_rmse_match else "FAIL"}] T3.6b RMSE numpy == sklearn')

# Mean predictor → R²≈0
y_mean_pred = np.full_like(y_val, y_train.mean())
r2_mean = core.r2_score_np(y_val, y_mean_pred)
ok_mean = abs(r2_mean) < 0.05
print(f'[{"PASS" if ok_mean else "FAIL"}] T3.6c Mean predictor → val R²={r2_mean:.6f} ≈ 0')

# Perfect prediction → R²=1
r2_perfect = core.r2_score_np(y_val, y_val)
ok_perfect = abs(r2_perfect - 1.0) < 1e-10
print(f'[{"PASS" if ok_perfect else "FAIL"}] T3.6d Perfect prediction → R²={r2_perfect}')

Train R² numpy=0.698405  sklearn=0.698405
Val   R² numpy=0.678527  sklearn=0.678527
Train RMSE numpy=100.0298  sklearn=100.0298
Val   RMSE numpy=100.3817  sklearn=100.3817
[PASS] T3.6a R² numpy == sklearn
[PASS] T3.6b RMSE numpy == sklearn
[PASS] T3.6c Mean predictor → val R²=-0.000002 ≈ 0
[PASS] T3.6d Perfect prediction → R²=1.0


In [20]:
# ── T3.7: Determinism ────────────────────────────────────────────────────────
CHOSEN_LR  = 1.0 / L_real  # will be updated after LR sweep in Task 4
CHOSEN_TOL = 1e-6
CHOSEN_ITER = 100000

res1 = core.gd_fit(X_train, y_train, lr=CHOSEN_LR, max_iter=CHOSEN_ITER,
                   tol=CHOSEN_TOL, X_val=X_val, y_val=y_val)
res2 = core.gd_fit(X_train, y_train, lr=CHOSEN_LR, max_iter=CHOSEN_ITER,
                   tol=CHOSEN_TOL, X_val=X_val, y_val=y_val)

ok_det = np.array_equal(res1['w'], res2['w']) and res1['b'] == res2['b']
print(f'[{"PASS" if ok_det else "FAIL"}] T3.7 Two runs give identical weights')

W_SHA256 = hashlib.sha256(res1['w'].tobytes()).hexdigest()
print(f'SHA-256 of w: {W_SHA256}')

[PASS] T3.7 Two runs give identical weights
SHA-256 of w: a5ebd419f5868cb22c4d633aaceaa7281e1a3a25b1b7a7ef3a28112efe820ff6


**Outcome** *(fill in — cite actual numbers)*:
> *Done: Report gradient check relative errors, synthetic recovery errors, closed-form comparison (loss gap, R² match, max |Δw|), and whether all tests passed.*

---
## Task 4 — Learning Rate Sweep

**Expectation** *(fill in)*:
> *Done: Predict which LR values will converge vs. diverge based on theory. At what threshold does theory say it diverges?*

In [21]:
# ── T4: LR sweep ────────────────────────────────────────────────────────────
L = L_real
lr_candidates = [
    1e-4,
    1e-3,
    1e-2,
    0.1,
    round(1.0 / L, 8),       # 1/L  (boundary: monotone)
    round(1.5 / L, 8),       # 1.5/L (stable, faster)
    round(1.9 / L, 8),       # 1.9/L (near boundary)
    round(2.0 / L, 8),       # 2/L   (theory: boundary, may oscillate)
    round(2.1 / L, 8),       # 2.1/L (theory: diverges)
]

print(f'L = {L:.4f}  |  1/L = {1/L:.6f}  |  2/L = {2/L:.6f}\n')
print(f'{"LR":>12}  {"Iters":>8}  {"Train Loss":>12}  {"Val R²":>10}  {"Status":<15}')
print('-' * 65)

sweep_results = []
for lr_val in lr_candidates:
    res = core.gd_fit(
        X_train, y_train,
        lr=lr_val, max_iter=20000, tol=1e-8, gtol=1e-10,
        X_val=X_val, y_val=y_val
    )
    y_vp = core.predict(X_val, res['w'], res['b'])
    r2   = core.r2_score_np(y_val, y_vp) if res['status'] != 'diverged' else float('nan')
    print(f'{lr_val:>12.6f}  {res["n_iter"]:>8d}  '
          f'{res["final_train_loss"]:>12.4f}  {r2:>10.6f}  {res["status"]:<15}')
    sweep_results.append({
        'lr': lr_val, 'n_iter': res['n_iter'],
        'train_loss': res['final_train_loss'],
        'val_r2': r2, 'status': res['status']
    })

L = 2.4219  |  1/L = 0.412897  |  2/L = 0.825793

          LR     Iters    Train Loss      Val R²  Status         
-----------------------------------------------------------------


    0.000100     20000    10693.1935    0.339534  not_converged  


    0.001000     20000     6649.6824    0.582267  not_converged  


    0.010000     20000     5169.5854    0.664930  not_converged  


    0.100000     20000     5004.6579    0.678464  not_converged  


    0.412897     11215     5003.8464    0.678630  converged      


    0.619345     12157     5003.5579    0.678621  converged      


    0.784504     11763     5003.4356    0.678614  converged      


    0.825793     20000    25620.2075   -0.614973  not_converged  
    0.867083        76  40367611739.0942         nan  diverged       


In [22]:
# ── T4 continued: choose final LR and run the definitive fit ────────────────
# Choose the largest stable (non-diverged) lr; prefer 1.5/L as it converges
# faster than 1/L without oscillating.
# Select only converged runs with lr <= 1.5/L (ensuring stable convergence, not boundary stalling)
converged_candidates = [r for r in sweep_results if r['status'] == 'converged' and r['lr'] <= 1.5 / L + 1e-9]
CHOSEN_LR = max(converged_candidates, key=lambda r: r['lr'])['lr']
assert CHOSEN_LR * L_real < 2.0, f'Chosen LR ({CHOSEN_LR}) violates stability bound 2/L ({2.0/L_real})'

print(f'\nChosen LR: {CHOSEN_LR:.8f}  (= {CHOSEN_LR*L:.2f}/L)')

CHOSEN_TOL  = 1e-6
CHOSEN_ITER = 200000

FINAL_RES = core.gd_fit(
    X_train, y_train,
    lr=CHOSEN_LR, max_iter=CHOSEN_ITER, tol=CHOSEN_TOL,
    X_val=X_val, y_val=y_val, record_every=100
)
W_FINAL = FINAL_RES['w']
B_FINAL = FINAL_RES['b']

y_tr_pred_final = core.predict(X_train, W_FINAL, B_FINAL)
y_vl_pred_final = core.predict(X_val,   W_FINAL, B_FINAL)

R2_TR  = core.r2_score_np(y_train, y_tr_pred_final)
R2_VL  = core.r2_score_np(y_val,   y_vl_pred_final)
RMSE_TR = core.rmse_np(y_train, y_tr_pred_final)
RMSE_VL = core.rmse_np(y_val,   y_vl_pred_final)

print(f'\n=== FINAL MODEL METRICS ===')
print(f'Train R²   : {R2_TR:.6f}')
print(f'Val   R²   : {R2_VL:.6f}')
print(f'Train RMSE : {RMSE_TR:.4f}')
print(f'Val   RMSE : {RMSE_VL:.4f}')
print(f'R²  gap    : {R2_TR - R2_VL:.6f}')
print(f'RMSE gap   : {RMSE_TR - RMSE_VL:.4f}')
print(f'Iterations : {FINAL_RES["n_iter"]}  Status: {FINAL_RES["status"]}')

W_SHA256_FINAL = hashlib.sha256(W_FINAL.tobytes()).hexdigest()
print(f'\nSHA-256(w) : {W_SHA256_FINAL}')


Chosen LR: 0.61934503  (= 1.50/L)



=== FINAL MODEL METRICS ===
Train R²   : 0.698181
Val   R²   : 0.678137
Train RMSE : 100.0670
Val   RMSE : 100.4427
R²  gap    : 0.020044
RMSE gap   : -0.3756
Iterations : 2389  Status: converged

SHA-256(w) : 8536e02b0ec993ce09bdf858ad70cb9e83e37609149d8808dc856fba59f0c6f4


**Outcome** *(fill in — cite actual numbers)*:
> *Done: State which LR was chosen and why. Report the val R² and RMSE. Does the theoretical threshold match what the sweep observed?*

---
## Task 5 — Ablation Study

**Expectation** *(fill in)*:
> *Done: Which features do you think will help most? Do you expect `atemp` to add anything given its high correlation with `temp`?*

In [23]:
# ── T5: Ablation table ───────────────────────────────────────────────────────
# Helper: closed-form R² for a given transform
def ablation_eval(params_override: dict, label: str):
    """Fit preprocessor variant, closed-form lstsq, return val R²."""
    # We use lstsq here, labelled as 'closed-form ablation'
    params_ab = core.fit_preprocessor(train_df, **params_override.get('preproc_kwargs', {}))
    # Apply any column drops via post-processing
    X_tr_ab, names_ab = core.transform(train_df, params_ab)
    X_vl_ab, _        = core.transform(val_df,   params_ab)

    # Optional post-transform column exclusions
    drop_prefixes = params_override.get('drop_prefixes', [])
    if drop_prefixes:
        keep = [i for i, n in enumerate(names_ab)
                if not any(n.startswith(pf) for pf in drop_prefixes)]
        X_tr_ab = X_tr_ab[:, keep]
        X_vl_ab = X_vl_ab[:, keep]

    X_aug_ab = np.hstack([X_tr_ab, np.ones((X_tr_ab.shape[0], 1))])
    theta_ab, _, _, _ = np.linalg.lstsq(X_aug_ab, y_train, rcond=None)
    w_ab, b_ab = theta_ab[:-1], float(theta_ab[-1])

    y_vl_ab = core.predict(X_vl_ab, w_ab, b_ab)
    r2_ab   = core.r2_score_np(y_val, y_vl_ab)
    rmse_ab = core.rmse_np(y_val, y_vl_ab)
    print(f'  {label:<45} val R²={r2_ab:.6f}  RMSE={rmse_ab:.3f}  p={X_tr_ab.shape[1]}')
    return r2_ab

print('=== ABLATION TABLE (closed-form, val set) ===')
print(f'{"Variant":<45} {"Val R²":>10}  {"RMSE":>8}  {"p":>5}')
print('-' * 75)

ablation_results = {}

# Baseline: all features (already computed above)
X_aug_base = np.hstack([X_train, np.ones((X_train.shape[0], 1))])
theta_base, _, _, _ = np.linalg.lstsq(X_aug_base, y_train, rcond=None)
w_base, b_base = theta_base[:-1], float(theta_base[-1])
y_vl_base = core.predict(X_val, w_base, b_base)
r2_base = core.r2_score_np(y_val, y_vl_base)
rmse_base = core.rmse_np(y_val, y_vl_base)
print(f'  {"Baseline (all features)":<45} val R²={r2_base:.6f}  RMSE={rmse_base:.3f}  p={X_train.shape[1]}')
ablation_results['Baseline'] = r2_base

# Without atemp
ablation_results['No atemp'] = ablation_eval(
    {'drop_prefixes': ['atemp']}, 'No atemp')

# Without days_since_start
ablation_results['No days_since_start'] = ablation_eval(
    {'drop_prefixes': ['days_since_start']}, 'No days_since_start')

# Without windspeed flag
ablation_results['No windspeed flag'] = ablation_eval(
    {'preproc_kwargs': {'add_windspeed_flag': False}}, 'No windspeed flag')

# Without yr (binary)
ablation_results['No yr'] = ablation_eval(
    {'drop_prefixes': ['yr']}, 'No yr')

# Without holiday
ablation_results['No holiday'] = ablation_eval(
    {'drop_prefixes': ['holiday']}, 'No holiday')

# Without workingday
ablation_results['No workingday'] = ablation_eval(
    {'drop_prefixes': ['workingday']}, 'No workingday')

print('\n=== Summary ===')
for k, v in sorted(ablation_results.items(), key=lambda x: -x[1]):
    delta = v - r2_base
    print(f'  {k:<35}  val R²={v:.6f}  Δ={delta:+.6f}')

=== ABLATION TABLE (closed-form, val set) ===
Variant                                           Val R²      RMSE      p
---------------------------------------------------------------------------
  Baseline (all features)                       val R²=0.678527  RMSE=100.382  p=54
  No atemp                                      val R²=0.677939  RMSE=100.473  p=53
  No days_since_start                           val R²=0.678606  RMSE=100.369  p=53


  No windspeed flag                             val R²=0.678451  RMSE=100.394  p=53
  No yr                                         val R²=0.678638  RMSE=100.364  p=53
  No holiday                                    val R²=0.678527  RMSE=100.382  p=53
  No workingday                                 val R²=0.678527  RMSE=100.382  p=53

=== Summary ===
  No yr                                val R²=0.678638  Δ=+0.000110
  No days_since_start                  val R²=0.678606  Δ=+0.000079
  No holiday                           val R²=0.678527  Δ=+0.000000
  No workingday                        val R²=0.678527  Δ=+0.000000
  Baseline                             val R²=0.678527  Δ=+0.000000
  No windspeed flag                    val R²=0.678451  Δ=-0.000077
  No atemp                             val R²=0.677939  Δ=-0.000588


**Outcome** *(fill in — cite actual numbers)*:
> *Done: State which feature set was chosen and why based on the ablation table. Report the Δ R² for removing each feature.*

---
## Task 6 — Diagnostics & Residual Analysis

**Expectation** *(fill in)*:
> *Done: What patterns do you expect in the residuals? E.g., systematic under-prediction at peak hours?*

In [24]:
# ── T6.1: Key metrics ────────────────────────────────────────────────────────
y_vl_pf = y_vl_pred_final  # val predictions from final GD model

neg_pct = 100.0 * (y_vl_pf < 0).mean()
r2_before_clip = core.r2_score_np(y_val, y_vl_pf)
r2_after_clip  = core.r2_score_np(y_val, np.clip(y_vl_pf, 0, None))

print(f'Train R²          : {R2_TR:.6f}')
print(f'Val   R²          : {R2_VL:.6f}')
print(f'Train RMSE        : {RMSE_TR:.4f}')
print(f'Val   RMSE        : {RMSE_VL:.4f}')
print(f'R² gap (tr-val)   : {R2_TR - R2_VL:.6f}')
print(f'Negative preds    : {neg_pct:.2f}%')
print(f'R² before clip    : {r2_before_clip:.6f}')
print(f'R² after clip @0  : {r2_after_clip:.6f}')

Train R²          : 0.698181
Val   R²          : 0.678137
Train RMSE        : 100.0670
Val   RMSE        : 100.4427
R² gap (tr-val)   : 0.020044
Negative preds    : 9.55%
R² before clip    : 0.678137
R² after clip @0  : 0.696571


In [25]:
# ── T6.2: Residual plots ─────────────────────────────────────────────────────
residuals = y_val - y_vl_pred_final
val_hr      = val_df['hr'].values
val_weekday = val_df['weekday'].values

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Residuals vs predicted
axes[0,0].scatter(y_vl_pred_final, residuals, alpha=0.15, s=5, color='steelblue')
axes[0,0].axhline(0, color='red', lw=1)
axes[0,0].set_xlabel('Predicted cnt'); axes[0,0].set_ylabel('Residual')
axes[0,0].set_title('Residuals vs Predicted')

# Residuals vs hour (box-like aggregation)
res_by_hr = pd.DataFrame({'hr': val_hr, 'res': residuals})
hr_means  = res_by_hr.groupby('hr')['res'].mean()
axes[0,1].bar(hr_means.index, hr_means.values, color='coral')
axes[0,1].axhline(0, color='black', lw=1)
axes[0,1].set_xlabel('Hour'); axes[0,1].set_ylabel('Mean Residual')
axes[0,1].set_title('Mean Residual by Hour')

# Residuals vs weekday
res_by_wd = pd.DataFrame({'wd': val_weekday, 'res': residuals}).groupby('wd')['res'].mean()
axes[0,2].bar(res_by_wd.index, res_by_wd.values, color='mediumorchid')
axes[0,2].axhline(0, color='black', lw=1)
axes[0,2].set_xlabel('Weekday'); axes[0,2].set_ylabel('Mean Residual')
axes[0,2].set_title('Mean Residual by Weekday')

# Convergence plot (log scale)
if FINAL_RES['train_losses']:
    iters_tr, losses_tr = zip(*FINAL_RES['train_losses'])
    axes[1,0].semilogy(iters_tr, losses_tr, color='blue',  label='Train', lw=1)
    if FINAL_RES['val_losses']:
        iters_vl, losses_vl = zip(*FINAL_RES['val_losses'])
        axes[1,0].semilogy(iters_vl, losses_vl, color='orange', label='Val',   lw=1)
    axes[1,0].set_xlabel('Iteration'); axes[1,0].set_ylabel('MSE Loss (log)')
    axes[1,0].set_title('Convergence Curve')
    axes[1,0].legend()

# Actual vs predicted
axes[1,1].scatter(y_val, y_vl_pred_final, alpha=0.1, s=4, color='green')
lim = max(y_val.max(), y_vl_pred_final.max())
axes[1,1].plot([0, lim], [0, lim], 'r--', lw=1)
axes[1,1].set_xlabel('Actual cnt'); axes[1,1].set_ylabel('Predicted cnt')
axes[1,1].set_title(f'Actual vs Predicted (val R²={R2_VL:.4f})')

# Top ±10 weights
top_pos = np.argsort(W_FINAL)[-10:][::-1]
top_neg = np.argsort(W_FINAL)[:10]
top_idx = list(top_pos) + list(top_neg)
top_vals = W_FINAL[top_idx]
top_lbls = [FEATURE_NAMES[i] for i in top_idx]
colors = ['steelblue' if v >= 0 else 'coral' for v in top_vals]
axes[1,2].barh(range(len(top_vals)), top_vals, color=colors)
axes[1,2].set_yticks(range(len(top_vals)))
axes[1,2].set_yticklabels(top_lbls, fontsize=7)
axes[1,2].axvline(0, color='black', lw=1)
axes[1,2].set_title('Top ±10 Weights\n(collinearity may make some unreliable)')

fig.tight_layout()
fig.savefig(PLOTS_DIR / '01_gd_diagnostics.png', dpi=100)
plt.close(fig)
print('[SAVED] plots/01_gd_diagnostics.png')

print('\nTop +10 weights (largest positive):')
for i in top_pos:
    print(f'  {FEATURE_NAMES[i]:<30} {W_FINAL[i]:+.4f}')
print('\nTop -10 weights (largest negative):')
for i in top_neg:
    print(f'  {FEATURE_NAMES[i]:<30} {W_FINAL[i]:+.4f}')

[SAVED] plots/01_gd_diagnostics.png

Top +10 weights (largest positive):
  hr_17                          +376.1532
  hr_18                          +343.0066
  hr_8                           +309.4217
  hr_19                          +228.3771
  hr_16                          +212.2798
  hr_12                          +161.6871
  hr_15                          +157.3562
  hr_13                          +155.4630
  hr_7                           +152.9998
  hr_9                           +151.1412

Top -10 weights (largest negative):
  weather_3                      -68.2181
  hr_3                           -53.4517
  hr_4                           -52.5267
  hr_2                           -39.2669
  hr_5                           -35.8034
  hr_1                           -27.9123
  hum                            -15.2562
  weather_2                      -12.1405
  mnth_12                        -10.4538
  is_windspeed_zero              -8.3849


**Outcome** *(fill in — cite actual numbers)*:
> *Done: Describe the residual patterns you observed. Are there systematic biases at certain hours? Heteroscedasticity? What does this imply for Phase 2?*

---
## Task 7 — Artifact Save & Round-Trip Test

**Expectation** *(fill in)*:
> *Done: What should the artifact round-trip test verify? Why is it important that Phase 2 can load and reproduce identical predictions?*

In [26]:
# ── T7: Save artifacts ───────────────────────────────────────────────────────
W_SHA256_FINAL = core.save_artifacts(
    path_stem=ART_STEM,
    w=W_FINAL,
    b=B_FINAL,
    feature_names=FEATURE_NAMES,
    params=PREP_PARAMS,
    seed=SEED,
    lr=CHOSEN_LR,
    tol=CHOSEN_TOL,
    max_iter=CHOSEN_ITER,
    n_iter=FINAL_RES['n_iter'],
    final_train_loss=FINAL_RES['final_train_loss'],
    final_val_loss=FINAL_RES['final_val_loss'],
    train_idx=TRAIN_IDX,
    val_idx=VAL_IDX,
)
print(f'[SAVED] {ART_STEM}.npz')
print(f'[SAVED] {ART_STEM}.json')
print(f'SHA-256(w) : {W_SHA256_FINAL}')

# ── T7: Round-trip test (T3.8) ────────────────────────────────────────────────
w_rt, b_rt, meta_rt, _, _ = core.load_artifacts(ART_STEM)
y_vl_rt = core.predict(X_val, w_rt, b_rt)
r2_rt   = core.r2_score_np(y_val, y_vl_rt)
ok_rt   = abs(r2_rt - R2_VL) < 1e-9
print(f'\n[{"PASS" if ok_rt else "FAIL"}] T3.8 Artifact round-trip: '
      f'loaded val R²={r2_rt:.8f}  original R²={R2_VL:.8f}')
sha_check = hashlib.sha256(w_rt.tobytes()).hexdigest()
ok_sha = (sha_check == meta_rt['w_sha256'])
print(f'[{"PASS" if ok_sha else "FAIL"}] SHA-256 matches stored value')

[SAVED] gradient_descent_artifacts.npz
[SAVED] gradient_descent_artifacts.json
SHA-256(w) : 8536e02b0ec993ce09bdf858ad70cb9e83e37609149d8808dc856fba59f0c6f4

[PASS] T3.8 Artifact round-trip: loaded val R²=0.67813650  original R²=0.67813650
[PASS] SHA-256 matches stored value


**Outcome** *(fill in)*:
> *Done: Confirm the round-trip passed. State the SHA-256 of the final weights — this is the value your team should record for reproducibility.*

---
## Task 8 — Test: Notebook Health

In [27]:
# ── T3.9 (run externally): nbconvert health check ────────────────────────────
# This cell just prints instructions; the actual test is run in the shell:
#   python -m jupyter nbconvert --to notebook --execute gradient_descent.ipynb --output phase1_executed.ipynb
print('Notebook health check should be run externally with:')
print('  python -m jupyter nbconvert --to notebook --execute gradient_descent.ipynb')
print('The cell above this is the last code cell — if you are reading this,'
      ' execution reached the end without error.')

Notebook health check should be run externally with:
  python -m jupyter nbconvert --to notebook --execute gradient_descent.ipynb
The cell above this is the last code cell — if you are reading this, execution reached the end without error.


---
## Task 9 (Bonus) — Asymmetric Loss

**Expectation** *(fill in)*:
> *Done: Under the asymmetric loss (penalty 3× for under-predictions), how do you expect the residual distribution to shift relative to MSE?*

In [28]:
# ── T9.1: Gradient check for asymmetric loss ─────────────────────────────────
gc_asym_rand = core.gradient_check(
    X_r, y_r, w_r, b_r,
    grad_fn=core.asymmetric_grad,
    loss_fn=lambda X, y, w, b: core.asymmetric_loss(X, y, w, b),
)
ok_gc_asym = gc_asym_rand['all_pass']
print(f'[{"PASS" if ok_gc_asym else "FAIL"}] T9.1 Asymmetric grad check  '
      f'max_rel_w={gc_asym_rand["max_rel_error_w"]:.2e}  '
      f'max_rel_b={gc_asym_rand["max_rel_error_b"]:.2e}')

[PASS] T9.1 Asymmetric grad check  max_rel_w=4.32e-11  max_rel_b=1.74e-12


In [29]:
# ── T9.2: Fit asymmetric model ───────────────────────────────────────────────
# lr must be <= 1/(3L) for asymmetric loss (curvature up to 3x larger)
LR_ASYM = 1.0 / (6.0 * L_real)
print(f'Asymmetric LR = 1/(6L) = {LR_ASYM:.8f}')

res_asym = core.gd_fit(
    X_train, y_train,
    lr=LR_ASYM,
    max_iter=200000,
    tol=1e-6,
    grad_fn=core.asymmetric_grad,
    loss_fn=core.asymmetric_loss,
    X_val=X_val, y_val=y_val,
    record_every=500
)

W_ASYM = res_asym['w']
B_ASYM = res_asym['b']

y_vl_asym = core.predict(X_val, W_ASYM, B_ASYM)
y_tr_asym = core.predict(X_train, W_ASYM, B_ASYM)

print(f'Asym model: iters={res_asym["n_iter"]}  status={res_asym["status"]}')

Asymmetric LR = 1/(6L) = 0.06881611


Asym model: iters=5975  status=converged


In [30]:
# ── T9.3: Monotonicity check for asymmetric GD ───────────────────────────────
res_asym_mono = core.gd_fit(
    X_train, y_train,
    lr=LR_ASYM, max_iter=500, tol=0, gtol=0, record_every=1,
    grad_fn=core.asymmetric_grad,
    loss_fn=core.asymmetric_loss,
)
losses_asym_seq = [v for _, v in res_asym_mono['train_losses']]
diffs_asym = np.diff(losses_asym_seq)
ok_mono_asym = bool(np.all(diffs_asym <= 1e-10))
print(f'[{"PASS" if ok_mono_asym else "FAIL"}] T9.3 Asym loss monotone at lr=1/(6L)  '
      f'max increase={diffs_asym.max():.2e}')

[PASS] T9.3 Asym loss monotone at lr=1/(6L)  max increase=-8.66e-01


In [31]:
# ── T9.4: Comparison MSE vs Asymmetric ───────────────────────────────────────
# Under-prediction rate (e = y - yhat > 0)
underpred_mse  = 100.0 * (y_val - y_vl_pred_final > 0).mean()
underpred_asym = 100.0 * (y_val - y_vl_asym > 0).mean()

mean_res_mse  = (y_val - y_vl_pred_final).mean()
mean_res_asym = (y_val - y_vl_asym).mean()

r2_mse  = core.r2_score_np(y_val, y_vl_pred_final)
r2_asym = core.r2_score_np(y_val, y_vl_asym)

asym_cost_mse  = core.asymmetric_loss(X_val, y_val, W_FINAL, B_FINAL)
asym_cost_asym = core.asymmetric_loss(X_val, y_val, W_ASYM,  B_ASYM)

print('=== MSE model vs Asymmetric model (val set) ===')
print(f'                       {"MSE model":>15}   {"Asym model":>15}')
print(f'Under-prediction rate  {underpred_mse:>14.2f}%   {underpred_asym:>14.2f}%')
print(f'Mean residual          {mean_res_mse:>15.4f}   {mean_res_asym:>15.4f}')
print(f'Val R²                 {r2_mse:>15.6f}   {r2_asym:>15.6f}')
print(f'Asymmetric cost (val)  {asym_cost_mse:>15.4f}   {asym_cost_asym:>15.4f}')
print('\n→ Phase 1→2 artifact uses MSE weights (as per plan).')
print('  Team should decide if asymmetric weights are preferable and document justification.')

=== MSE model vs Asymmetric model (val set) ===
                             MSE model        Asym model
Under-prediction rate           46.69%            32.69%
Mean residual                  -1.7974          -43.3625
Val R²                        0.678137          0.596060
Asymmetric cost (val)       20602.2158        17448.2716

→ Phase 1→2 artifact uses MSE weights (as per plan).
  Team should decide if asymmetric weights are preferable and document justification.


**Outcome** *(fill in — cite actual numbers)*:
> *Done: Report the under-prediction rate for both models, the mean residuals, the R² values, and whether the asymmetric model wins on asymmetric cost. Justify the choice of which weights to carry forward to Phase 2.*

---
## Final Summary for the Agent

> This cell is printed by the agent after all tasks complete — DO NOT edit until you have seen the printed values.

In [32]:
# ── FINAL CHECKLIST ──────────────────────────────────────────────────────────
print('=' * 60)
print('PHASE 1 — FINAL CHECKLIST')
print('=' * 60)
print(f'Seed          : {SEED}')
print(f'Train rows    : {len(train_df)}')
print(f'Val rows      : {len(val_df)}')
print(f'Features (p)  : {X_train.shape[1]}')
print(f'Chosen LR     : {CHOSEN_LR:.8f}  (= {CHOSEN_LR * L_real:.3f}/L)')
print(f'L             : {L_real:.4f}')
print(f'1/L           : {1/L_real:.6f}')
print(f'2/L           : {2/L_real:.6f}')
print(f'Iterations    : {FINAL_RES["n_iter"]}  ({FINAL_RES["status"]})')
print(f'Train R²      : {R2_TR:.6f}')
print(f'Val   R²      : {R2_VL:.6f}')
print(f'Train RMSE    : {RMSE_TR:.4f}')
print(f'Val   RMSE    : {RMSE_VL:.4f}')
print(f'R² gap        : {R2_TR - R2_VL:.6f}')
print(f'SHA-256(w)    : {W_SHA256_FINAL}')
print()
print('TEST SUITE:')
checks = [
    ('T3.1  Gradient check (MSE)',             ok_gc_rand and ok_gc_real),
    ('T3.2  Synthetic recovery',               ok_clean and ok_noisy),
    ('T3.3  Closed-form comparison',           ok_loss and ok_r2),
    ('T3.4  Monotonicity',                     ok_mono and ok_div),
    ('T3.5  Stopping logic',                   ok_nc and ok_conv),
    ('T3.6  Metrics (numpy vs sklearn)',       ok_r2_match and ok_rmse_match and ok_mean and ok_perfect),
    ('T3.7  Determinism',                      ok_det),
    ('T3.8  Artifact round-trip',              ok_rt and ok_sha),
    ('T3.9  Notebook health (external)',       Path('gradient_descent.ipynb').exists()),  # run separately
    ('T9.1  Asym grad check',                  ok_gc_asym),
    ('T9.3  Asym monotonicity',                ok_mono_asym),
]
all_pass = True
for name, result in checks:
    status = 'PASS' if result else 'FAIL'
    if not result: all_pass = False
    print(f'  [{status}] {name}')

print()
print(f'Overall: {"ALL TESTS PASSED" if all_pass else "SOME TESTS FAILED — review output above"}')
print('=' * 60)

PHASE 1 — FINAL CHECKLIST
Seed          : 42648
Train rows    : 8708
Val rows      : 2178
Features (p)  : 54
Chosen LR     : 0.61934503  (= 1.500/L)
L             : 2.4219
1/L           : 0.412897
2/L           : 0.825793
Iterations    : 2389  (converged)
Train R²      : 0.698181
Val   R²      : 0.678137
Train RMSE    : 100.0670
Val   RMSE    : 100.4427
R² gap        : 0.020044
SHA-256(w)    : 8536e02b0ec993ce09bdf858ad70cb9e83e37609149d8808dc856fba59f0c6f4

TEST SUITE:
  [PASS] T3.1  Gradient check (MSE)
  [PASS] T3.2  Synthetic recovery
  [PASS] T3.3  Closed-form comparison
  [PASS] T3.4  Monotonicity
  [PASS] T3.5  Stopping logic
  [PASS] T3.6  Metrics (numpy vs sklearn)
  [PASS] T3.7  Determinism
  [PASS] T3.8  Artifact round-trip
  [PASS] T3.9  Notebook health (external)
  [PASS] T9.1  Asym grad check
  [PASS] T9.3  Asym monotonicity

Overall: ALL TESTS PASSED
